# Phase 6 DAEAC Reliability-Weighted Prototype Bank
Runs the logging-only control and reliability-weighted global-prototype ablation. Checkpoints are selected only by source-validation Macro-F1.

## 1. Setup

In [ ]:
from pathlib import Path
import importlib.util
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/YOUR_ORG/Best-thesis-in-council.git'
BRANCH = 'main'
TRAIN_BASE_IF_MISSING = False
FULL_VARIANTS = ['logging_only', 'weighted_global']
RESUME_CHECKPOINTS = {'logging_only': None, 'weighted_global': None}
WANDB_ENABLED = False
WANDB_PROJECT = 'ecg-thesis'
WANDB_ENTITY = None
WANDB_GROUP = 'phase6-daeac-prototype-bank'

VARIANTS = {
    'logging_only': {
        'config': 'configs/phase6_daeac_proto_bank_logging.yaml',
        'prefix': 'daeac_proto_bank_logging',
        'output': Path('outputs/phase6_daeac_proto_bank_logging'),
    },
    'weighted_global': {
        'config': 'configs/phase6_daeac_proto_bank_weighted.yaml',
        'prefix': 'daeac_proto_bank_weighted',
        'output': Path('outputs/phase6_daeac_proto_bank_weighted'),
    },
}
assert set(FULL_VARIANTS).issubset(VARIANTS), FULL_VARIANTS

def run(args, cwd=None):
    print('+', ' '.join(str(value) for value in args))
    subprocess.check_call([str(value) for value in args], cwd=cwd)

working = Path('/kaggle/working')
repo_candidates = [path.parent for path in working.glob('**/ecg_thesis') if (path / 'scripts/check_repo.py').exists()]
if repo_candidates:
    REPO_DIR = repo_candidates[0]
else:
    assert 'YOUR_ORG' not in REPO_URL, 'Set REPO_URL before cloning.'
    REPO_ROOT = working / 'Best-thesis-in-council'
    run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_ROOT])
    REPO_DIR = REPO_ROOT / 'ecg_thesis'
os.chdir(REPO_DIR)
print('Project:', REPO_DIR)

## 2. Dependencies

In [ ]:
required = {'yaml': 'PyYAML', 'numpy': 'numpy', 'torch': 'torch', 'sklearn': 'scikit-learn'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    run([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('Dependencies ready')

## 3. Locate and copy data/checkpoint

In [ ]:
input_root = Path('/kaggle/input')
data_dest = REPO_DIR / 'data/processed/phase6_daeac_paper'
data_dest.mkdir(parents=True, exist_ok=True)
required_npz = ['mitdb_ds1_daeac.npz', 'mitdb_ds2_first5_unlabeled_daeac.npz', 'mitdb_ds2_daeac.npz']
optional_npz = ['incart_all_daeac.npz', 'svdb_all_daeac.npz']

def find_input(name):
    matches = sorted(input_root.glob(f'**/{name}'))
    return matches[0] if matches else None

for name in required_npz:
    source = find_input(name)
    assert source is not None, f'Missing required Kaggle input: {name}'
    shutil.copy2(source, data_dest / name)
for name in optional_npz:
    source = find_input(name)
    if source is not None:
        shutil.copy2(source, data_dest / name)
print('Data:', sorted(path.name for path in data_dest.glob('*.npz')))

base_name = 'daeac_base_best.pt'
base_dest = REPO_DIR / 'outputs/phase6_daeac_paper/checkpoints' / base_name
base_source = find_input(base_name)
if base_source is not None:
    base_dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(base_source, base_dest)
elif TRAIN_BASE_IF_MISSING:
    run([sys.executable, 'scripts/phase6_daeac_paper/01_train_base.py', '--config', 'configs/phase6_daeac_paper.yaml'], cwd=REPO_DIR)
else:
    raise FileNotFoundError(f'Attach {base_name} or set TRAIN_BASE_IF_MISSING=True.')
assert base_dest.exists(), base_dest
print('Base checkpoint:', base_dest)

## 4. Static, unit, and strict protocol checks

In [ ]:
run([sys.executable, 'scripts/check_repo.py'], cwd=REPO_DIR)
run([sys.executable, '-m', 'unittest', 'tests.test_daeac_prototype_bank', 'tests.test_daeac_prototype_bank_training'], cwd=REPO_DIR)
prepare_config = VARIANTS['logging_only']['config']
run([sys.executable, 'scripts/phase6_daeac_proto_bank/00_prepare_after5.py', '--config', prepare_config], cwd=REPO_DIR)
for variant in FULL_VARIANTS:
    run([sys.executable, 'scripts/phase6_daeac_proto_bank/01_validate.py', '--config', VARIANTS[variant]['config'], '--strict', '--init-checkpoint', base_dest], cwd=REPO_DIR)

## 5. One-epoch smoke tests

In [ ]:
for variant in FULL_VARIANTS:
    item = VARIANTS[variant]
    smoke_output = Path(f"outputs/{item['prefix']}_smoke")
    run([
        sys.executable, 'scripts/phase6_daeac_proto_bank/02_train.py',
        '--config', item['config'], '--init-checkpoint', base_dest,
        '--epochs', '1', '--max-source-samples', '512', '--max-target-samples', '512', '--max-val-samples', '512',
        '--checkpoint-prefix', f"{item['prefix']}_smoke", '--output-dir', smoke_output,
    ], cwd=REPO_DIR)

## 6. Full adaptation

In [ ]:
for variant in FULL_VARIANTS:
    item = VARIANTS[variant]
    command = [sys.executable, 'scripts/phase6_daeac_proto_bank/02_train.py', '--config', item['config'], '--init-checkpoint', base_dest]
    if RESUME_CHECKPOINTS.get(variant):
        command += ['--resume-checkpoint', RESUME_CHECKPOINTS[variant]]
    if WANDB_ENABLED:
        command += ['--wandb', '--wandb-project', WANDB_PROJECT, '--wandb-group', WANDB_GROUP, '--wandb-run-name', item['prefix']]
        if WANDB_ENTITY:
            command += ['--wandb-entity', WANDB_ENTITY]
    run(command, cwd=REPO_DIR)

## 7. Post-training evaluation

In [ ]:
for variant in FULL_VARIANTS:
    item = VARIANTS[variant]
    best_checkpoint = REPO_DIR / item['output'] / 'checkpoints' / f"{item['prefix']}_best.pt"
    assert best_checkpoint.exists(), best_checkpoint
    command = [
        sys.executable, 'scripts/phase6_daeac_proto_bank/03_eval.py',
        '--config', item['config'], '--checkpoint', best_checkpoint, '--method-name', item['prefix'], '--dataset', 'all',
    ]
    if WANDB_ENABLED:
        command += ['--wandb', '--wandb-project', WANDB_PROJECT, '--wandb-group', WANDB_GROUP, '--wandb-run-name', f"{item['prefix']}-best-eval"]
        if WANDB_ENTITY:
            command += ['--wandb-entity', WANDB_ENTITY]
    run(command, cwd=REPO_DIR)
run([sys.executable, 'scripts/phase6_daeac_proto_bank/04_make_report.py', '--config', VARIANTS['weighted_global']['config']], cwd=REPO_DIR)

## 8. Persist outputs

In [ ]:
bundle = Path('/kaggle/working/phase6_daeac_proto_bank_bundle')
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir(parents=True)
for variant in FULL_VARIANTS:
    item = VARIANTS[variant]
    source = REPO_DIR / item['output']
    if source.exists():
        shutil.copytree(source, bundle / source.name)
archive = shutil.make_archive(str(bundle), 'zip', bundle)
print('Created:', archive)